In [3]:
import polars as pl

raw_data = pl.scan_csv("../data/normalized/tracks.csv")  # type: ignore[reportArgumentType]

metrics = raw_data.select(
    [
        pl.len().alias("total_entries"),
        (pl.col("popularity") == 0).sum().alias("popularity_0"),
        (pl.col("popularity") < 15).sum().alias("popularity_under_15"),
        (pl.col("popularity") < 30).sum().alias("popularity_under_30"),
        (pl.col("popularity") < 50).sum().alias("popularity_under_50"),
        (pl.col("tempo") == 0).sum().alias("tempo_0"),
        (pl.col("tempo") < 60).sum().alias("tempo_under_60"),
        (pl.col("tempo") > 200).sum().alias("tempo_over_200"),
    ]
).collect()

print(metrics)

shape: (1, 8)
┌────────────┬────────────┬────────────┬────────────┬────────────┬─────────┬───────────┬───────────┐
│ total_entr ┆ popularity ┆ popularity ┆ popularity ┆ popularity ┆ tempo_0 ┆ tempo_und ┆ tempo_ove │
│ ies        ┆ _0         ┆ _under_15  ┆ _under_30  ┆ _under_50  ┆ ---     ┆ er_60     ┆ r_200     │
│ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ u32     ┆ ---       ┆ ---       │
│ u32        ┆ u32        ┆ u32        ┆ u32        ┆ u32        ┆         ┆ u32       ┆ u32       │
╞════════════╪════════════╪════════════╪════════════╪════════════╪═════════╪═══════════╪═══════════╡
│ 89741      ┆ 9448       ┆ 17360      ┆ 40475      ┆ 68532      ┆ 157     ┆ 477       ┆ 451       │
└────────────┴────────────┴────────────┴────────────┴────────────┴─────────┴───────────┴───────────┘


In [9]:
from pathlib import Path

from spotify_data_analytics import CsvStorage

track_genres = pl.scan_csv("../data/normalized/track_genres.csv")  # type: ignore[reportArgumentType]
genres = pl.scan_csv("../data/normalized/genres.csv")  # type: ignore[reportArgumentType]

genre_counts = (
    track_genres.join(genres, on="genre_id", how="inner")
    .group_by("genre_name")
    .agg(unique_tracks=pl.col("track_id").n_unique())
    .sort("unique_tracks", descending=True)
    .collect()
)

print(genre_counts)

storage = CsvStorage(Path("../data/processed/genre_counts.csv"))
storage.save(genre_counts)

shape: (114, 2)
┌────────────┬───────────────┐
│ genre_name ┆ unique_tracks │
│ ---        ┆ ---           │
│ str        ┆ u32           │
╞════════════╪═══════════════╡
│ metalcore  ┆ 1000          │
│ mandopop   ┆ 1000          │
│ tango      ┆ 1000          │
│ electronic ┆ 1000          │
│ reggaeton  ┆ 1000          │
│ …          ┆ …             │
│ honky-tonk ┆ 981           │
│ dance      ┆ 965           │
│ german     ┆ 963           │
│ classical  ┆ 933           │
│ romance    ┆ 904           │
└────────────┴───────────────┘
